# Lesson 02 — Data Preparation & Cleaning
## From raw data to a trustworthy ML dataset

**Course:** Machine Learning Recovery  
**Level:** Beginner → intermediate foundations  
**Tools:** Python, pandas, NumPy, scikit-learn  
**Estimated study time:** 2–3 hours

---

## Learning outcomes

By the end of this lesson, you should be able to:

1. Explain why data preparation is necessary before model training.
2. Detect and investigate missing values.
3. Choose between dropping and imputing missing values.
4. Detect and handle duplicate records.
5. inspect and correct data types.
6. Standardize inconsistent categorical values.
7. Detect suspicious values and outliers without blindly deleting them.
8. Explain data leakage and avoid a common form of it.
9. Build a reproducible cleaning workflow with pandas.

**Working rule:** Do not change data just because a value looks unusual. Investigate first, make a reasoned decision, and document it.

## 1. The role of data preparation

A model learns patterns from the examples we provide. If those examples contain errors, inconsistent formats, duplicated records, or missing information, the model may learn misleading patterns.

Data preparation is not simply "delete nulls." It is a sequence of decisions:
- Understand what each column means.
- Check whether values are valid.
- Identify missing or duplicated records.
- Choose appropriate transformations.
- Keep a record of decisions so the work can be repeated.

### Example scenario

Imagine a housing dataset used to estimate house prices:

| House | Bedrooms | Area (m²) | Price ($) |
|---|---:|---:|---:|
| A | 3 | 120 | 50,000 |
| B | 4 | 180 | 75,000 |
| C | missing | 150 | 60,000 |
| D | 3 | 120 | 50,000 |
| E | 5 | missing | 90,000 |
| F | 2 | 80 | -500 |

Possible issues:
- House C has a missing bedroom count.
- House E has a missing area.
- House D may duplicate House A.
- House F has a suspicious negative price.

Notice the word **may**. A duplicate-looking row or unusual value must be investigated before changing it.

## 2. Missing-value mechanisms

Three commonly discussed mechanisms are:

- **MCAR — Missing Completely At Random:** The probability of a value being missing is unrelated to observed or unobserved data. Example: a random technical failure.
- **MAR — Missing At Random:** Missingness is related to other observed variables. Example: a survey response is more often missing for a particular age group.
- **MNAR — Missing Not At Random:** Missingness is related to the missing value itself or an unobserved factor. Example: people with very high incomes may be less likely to report income.

These mechanisms matter because a simple mean or median replacement can introduce bias when missingness is systematic.

## 3. Set up the environment


In [ ]:
%pip install pandas numpy scikit-learn matplotlib


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## 4. Create and inspect a raw dataset

We will use a small **fictional teaching dataset**. It intentionally contains several data-quality issues so you can practice identifying them.

In real work, first read the data dictionary or ask the data owner what each column means. Never assume units or business rules.


In [ ]:
raw_data = {
    "customer_id": [101, 102, 103, 104, 104, 106, 107, 108, 109, 110, 111, 112],
    "age": [25, 34, np.nan, 29, 29, 41, 150, 23, 37, 31, 28, 45],
    "monthly_income": [500, 800, 650, np.nan, np.nan, 1200, 900, 450, 700, 620, 580, 1500],
    "city": ["Mogadishu", "Hargeisa", "mogadishu", "Bosaso ", "Bosaso ", "Kismayo", "Hargeisa", None, "MOGADISHU", "Garowe", "Garowe", "Kismayo"],
    "membership": ["Gold", "Silver", "Gold", "Bronze", "Bronze", "Gold", "Silver", "silver", "Gold ", "Bronze", None, "Gold"],
    "purchases": [5, 2, 3, 4, 4, 8, 1, 2, 6, 3, 2, 10],
    "annual_spend": [1200, 800, 950, 700, 700, 3000, 500, 600, 1600, 900, 750, 4200]
}

df_raw = pd.DataFrame(raw_data)
df = df_raw.copy()

df


In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst five rows:")
display(df.head())

print("\nBasic numeric summary:")
display(df.describe())


### How to inspect correctly

- `head()` shows the first rows; `tail()` shows the last rows.
- `shape` returns `(number_of_rows, number_of_columns)`.
- `dtypes` shows how pandas currently interprets each column.
- `describe()` summarizes numeric columns by default.
- `info()` gives a compact view of column types and non-null counts.

A dataset can load successfully and still contain poor-quality values. Loading is not validation.


In [ ]:
df.info()


## 5. Detect missing values

Use `isna()` or `isnull()` to identify missing values. They are equivalent for this use.

- `df.isna()` returns True/False for each cell.
- `df.isna().sum()` counts missing cells per column.
- `df.isna().mean()` gives the fraction missing per column.
- `df.isna().sum().sum()` counts all missing cells in the dataset.


In [ ]:
missing_count = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(1)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

display(missing_summary)
print("Total missing cells:", df.isna().sum().sum())


### Choosing what to do with missing values

There is no single correct method for every dataset.

**Option A — Drop rows**

Consider dropping rows when:
- Only a small number of rows are affected.
- The rows are not important or systematically different.
- Enough representative data remains.

Risks:
- You lose information.
- If missingness is systematic, you may create a biased dataset.

**Option B — Drop a column**

Consider dropping a column when:
- A very large proportion is missing.
- The column is not important to the task.
- It cannot be recovered or interpreted reliably.

Do not use a fixed percentage as an automatic rule. Context matters.

**Option C — Impute (fill) missing values**

Common simple methods:
- Numeric mean: useful for fairly symmetric data without strong outliers.
- Numeric median: often more robust when numeric data is skewed or has outliers.
- Categorical mode: fills with the most frequent category, but can over-represent that category.
- Constant such as `"Unknown"`: useful when missingness itself may carry information and the category is meaningful.

For some datasets, more advanced imputation or a missingness indicator may be appropriate.

### Important: fit preprocessing on training data only

When preparing data for machine learning, calculate imputation values using the training split only. If you calculate the mean/median using the full dataset before splitting, information from the test set can leak into training. We will practice a safe pipeline later in this lesson.


In [ ]:
# Example: inspect numeric distributions before choosing mean or median.
display(df[["age", "monthly_income", "purchases", "annual_spend"]].describe())

print("Income mean:", df["monthly_income"].mean())
print("Income median:", df["monthly_income"].median())


### Practical comparison: mean vs median

The mean is sensitive to extreme values. The median is the middle value after sorting and is generally less affected by a few extreme values.

Do not choose mean or median by habit. Inspect the distribution, understand the meaning of the feature, and consider why values are missing.


In [ ]:
# Make separate copies to compare methods. Do not overwrite the original yet.
income_mean_filled = df["monthly_income"].fillna(df["monthly_income"].mean())
income_median_filled = df["monthly_income"].fillna(df["monthly_income"].median())

comparison = pd.DataFrame({
    "original": df["monthly_income"],
    "mean_filled": income_mean_filled,
    "median_filled": income_median_filled
})

display(comparison)


## 6. Detect and investigate duplicates

A duplicate row is a repeated record. Duplicates can:
- Give repeated examples too much influence during training.
- Distort summaries and counts.
- Indicate an ingestion or data-entry issue.

Pandas tools:
- `duplicated()` marks duplicate rows (by default, later occurrences).
- `duplicated().sum()` counts them.
- `drop_duplicates()` removes them.

**Critical distinction:** A repeated customer ID is not automatically a duplicate row. One customer may legitimately have multiple transactions. You must know what one row represents.


In [ ]:
# Exact duplicate rows across all columns
print("Exact duplicate rows:", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values("customer_id"))

# Check repeated identifiers separately
print("Repeated customer IDs:", df["customer_id"].duplicated().sum())
display(df[df["customer_id"].duplicated(keep=False)].sort_values("customer_id"))


In this teaching dataset, the repeated row is identical across all columns. For this example, we will keep the first copy. In a real customer-transaction dataset, repeated customer IDs could be legitimate and should not be removed without checking the table's meaning.


In [ ]:
df_no_duplicates = df.drop_duplicates().copy()
print("Before:", df.shape)
print("After: ", df_no_duplicates.shape)


## 7. Correct data types

Data types affect calculations, sorting, memory use, and model preparation.

Examples:
- Age should be numeric.
- A date should be parsed as a datetime.
- A postal code or customer ID may be an identifier, not a number to calculate an average from.
- Categories such as city or membership are usually text/category data.

A column containing values such as `"25"`, `"unknown"`, and `"31"` may be read as text even though it represents a numeric measurement.


In [ ]:
type_example = pd.DataFrame({
    "age": ["25", "34", "unknown", "29"],
    "signup_date": ["2025-01-10", "2025-02-12", "not-a-date", "2025-04-01"]
})

print(type_example.dtypes)
display(type_example)


In [ ]:
# Convert age to numeric. Values that cannot be parsed become NaN.
type_example["age"] = pd.to_numeric(type_example["age"], errors="coerce")

# Convert dates. Invalid date strings become NaT.
type_example["signup_date"] = pd.to_datetime(
    type_example["signup_date"],
    errors="coerce"
)

print(type_example.dtypes)
display(type_example)


`errors="coerce"` is useful for detecting invalid entries because it converts them to missing values. But it does not solve the underlying data-quality issue. You should inspect which values failed conversion and decide what to do.


In [ ]:
# Find rows where the original age could not be converted.
invalid_age_mask = pd.to_numeric(
    type_example["age"], errors="coerce"
).isna()

# The converted column already contains NaN for the invalid value.
display(type_example[type_example["age"].isna()])


## 8. Standardize inconsistent categorical values

Categories that mean the same thing should use a consistent representation.

For example:
- `"Mogadishu"`
- `"mogadishu"`
- `"MOGADISHU "`

These may all refer to the same city, but a computer can treat them as different strings.

Common operations:
- `str.strip()` removes leading/trailing spaces.
- `str.lower()` converts text to lowercase.
- `replace()` maps known variations to a standard value.

Do not blindly standardize every string: capitalization may matter for names, codes, or language-specific text. Define a rule that fits the column.


In [ ]:
# Work on a copy after removing the exact duplicate.
df_clean = df_no_duplicates.copy()

for column in ["city", "membership"]:
    df_clean[column] = df_clean[column].str.strip().str.title()

display(df_clean[["city", "membership"]])
print("Unique cities:", df_clean["city"].dropna().unique())
print("Unique membership values:", df_clean["membership"].dropna().unique())


## 9. Detect invalid values and outliers

An **invalid value** violates a known rule. Example: a house price of -500 dollars when the dataset definition says prices cannot be negative.

An **outlier** is an observation unusually far from the rest. An outlier is not necessarily an error. A very high income or house price may be real.

Possible causes:
- Data-entry error.
- Measurement or sensor error.
- Rare but genuine event.
- Different population or process.

### Basic range checks

Use domain knowledge to define sensible constraints. For example, an age of 150 is suspicious in this customer dataset. Do not assume one universal age rule for every dataset.


In [ ]:
# Inspect suspicious age values using a domain rule for this example.
suspicious_age = df_clean[(df_clean["age"] < 0) | (df_clean["age"] > 100)]
display(suspicious_age)

# Inspect negative monthly income values, if any.
display(df_clean[df_clean["monthly_income"] < 0])


### IQR method for flagging numeric outliers

The Interquartile Range (IQR) is:

- Q1: 25th percentile.
- Q3: 75th percentile.
- IQR: Q3 − Q1.
- Lower fence: Q1 − 1.5 × IQR.
- Upper fence: Q3 + 1.5 × IQR.

Values outside these fences are flagged for investigation. This is a detection rule, not an instruction to delete every flagged row. It can be unsuitable for some distributions or small datasets.


In [ ]:
column = "annual_spend"

q1 = df_clean[column].quantile(0.25)
q3 = df_clean[column].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outlier_mask = (df_clean[column] < lower_fence) | (df_clean[column] > upper_fence)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)
print("Flagged rows:", outlier_mask.sum())

display(df_clean.loc[outlier_mask, ["customer_id", column]])


## 10. Data leakage — a critical ML concept

**Data leakage** happens when information that would not legitimately be available at prediction time gets into model training or evaluation. It can make test performance look much better than real-world performance.

Two common examples:

1. **Preprocessing leakage:** You calculate the mean, median, scaling values, or feature-selection decisions using the entire dataset before separating training and test data. The test set has influenced the training process.
2. **Target leakage:** A feature contains information that directly reveals the answer you are trying to predict. Example: using a "loan approved" status to predict whether that same loan will be approved.

### Safer workflow

1. Separate features and target.
2. Split into training and test sets.
3. Fit imputation, scaling, and other learned preprocessing on training data only.
4. Apply the already-fitted transformations to the test data.
5. Train on training data and evaluate on the untouched test data.

Scikit-learn Pipelines help keep these steps in the correct order.


In [ ]:
# Demonstration: safe imputation with a scikit-learn Pipeline.
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Use a built-in labeled dataset for this demonstration.
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer(as_frame=True)
X_demo = dataset.data
y_demo = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_demo
)

numeric_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

model_pipeline = Pipeline(steps=[
    ("preprocessing", numeric_preprocessing),
    ("model", LogisticRegression(max_iter=3000))
])

model_pipeline.fit(X_train, y_train)
print("Training score:", round(model_pipeline.score(X_train, y_train), 3))
print("Test score:", round(model_pipeline.score(X_test, y_test), 3))


The pipeline learns the median and scaling parameters from `X_train` during `.fit()`. When `.score()` receives `X_test`, it applies those learned transformations without fitting them again on the test data.

This example demonstrates a safe pattern. The score is not the focus of this lesson; understanding the separation is.


## 11. Build a cleaning audit

Professional data work should be explainable. Record what you changed and why. Do not silently overwrite raw data.

An audit log can include:
- Issue detected.
- Column or rows affected.
- Decision made.
- Reason for the decision.
- Number of values changed or rows removed.

Example:


In [ ]:
cleaning_log = [
    {
        "issue": "Exact duplicate row",
        "action": "Removed later exact duplicate",
        "reason": "All fields matched; this example treats each row as one customer record."
    },
    {
        "issue": "Inconsistent category formatting",
        "action": "Trimmed spaces and standardized title case",
        "reason": "Equivalent city and membership labels should have a consistent representation."
    }
]

pd.DataFrame(cleaning_log)


## 12. Final inspection

After cleaning, inspect the data again. Cleaning is not complete until you verify the result.

Questions to ask:
- Did the number of rows change? Why?
- Are missing values still present?
- Did any column unexpectedly change type?
- Are category values standardized?
- Did we accidentally remove legitimate observations?
- Can we explain every transformation?


In [ ]:
print("Raw shape:", df_raw.shape)
print("Cleaned shape (after exact duplicate removal and category formatting):", df_clean.shape)

print("\nMissing values after these steps:")
print(df_clean.isna().sum())

print("\nData types:")
print(df_clean.dtypes)

print("\nUnique category values:")
print("City:", df_clean["city"].dropna().unique())
print("Membership:", df_clean["membership"].dropna().unique())

display(df_clean.head(10))


# Lesson 02 — Practical assignment

Complete these tasks yourself in the cells below. Do not simply copy the demonstration cells.

### Part A — Missing data investigation
1. Calculate the number and percentage of missing values in every column of `df_raw`.
2. Identify which columns have missing values.
3. For `age`, calculate the median. Explain why median may be preferable to mean if a column contains extreme values.
4. Create a copy of `df_raw` and fill missing `age` values with the median. Do not modify `df_raw`.
5. Explain one situation where dropping rows would be a poor decision.

### Part B — Duplicate records
6. Count exact duplicate rows in `df_raw`.
7. Create a new DataFrame with exact duplicates removed.
8. Explain why repeated IDs are not always duplicate records.

### Part C — Data types and consistency
9. Display the data types in `df_raw`.
10. Standardize the `city` column by trimming spaces and making the text lowercase. Preserve missing values.
11. Display the unique city values before and after standardization.

### Part D — Validation and reasoning
12. Find rows where age is greater than 100.
13. Explain why an outlier should be investigated rather than automatically deleted.
14. In your own words, explain data leakage and why preprocessing should be fitted on training data only.

### Part E — Mini cleaning workflow
15. Create a new DataFrame from `df_raw`.
16. Remove exact duplicate rows only.
17. Standardize `city` and `membership`.
18. Fill missing `age` using the median calculated from this training exercise's data copy. State this is a simple classroom demonstration; in a real ML workflow the imputer must be fitted on the training split only.
19. Print the final shape, missing-value counts, and data types.
20. Write a short cleaning report explaining each decision.

## Assignment starter cells


In [ ]:
# Part A — Missing data investigation

# 1. Missing count and percentage:

# 2. Columns with missing values:

# 3. Median age:

# 4. Fill missing age in a COPY:

# 5. When might dropping rows be a poor decision?


In [ ]:
# Part B — Duplicate records

# 6. Count exact duplicates:

# 7. Remove exact duplicates into a new DataFrame:

# 8. Why are repeated IDs not always duplicate records?


In [ ]:
# Part C — Data types and consistency

# 9. Display data types:

# 10. Standardize city (lowercase + strip):

# 11. Compare unique values before and after:


In [ ]:
# Part D — Validation and reasoning

# 12. Find ages above 100:

# 13. Why investigate rather than automatically delete outliers?

# 14. Explain data leakage in your own words:


In [ ]:
# Part E — Mini cleaning workflow

# Build your own cleaned DataFrame step by step.
# Keep df_raw unchanged.

# 15. Create a copy:

# 16. Remove exact duplicate rows:

# 17. Standardize city and membership:

# 18. Impute missing age:

# 19. Print final shape, missing values, and dtypes:

# 20. Write a short cleaning report in a Markdown cell.


# Knowledge checkpoint

Answer in your own words:

1. What is the difference between a missing value and an invalid value?
2. When might median imputation be more appropriate than mean imputation?
3. Why should we investigate before dropping duplicate-looking records?
4. What is the difference between an outlier and an error?
5. What is data leakage?
6. Why must an imputer be fitted on training data rather than the full dataset?
7. What information should a cleaning log contain?

## Completion criteria

- Run every notebook code cell without errors.
- Complete all 20 practical tasks.
- Answer all 7 checkpoint questions.
- Submit the notebook and your short cleaning report for review.

**Next lesson:** Exploratory Data Analysis (EDA) — understanding distributions, relationships, correlations, and patterns before building models.

Do not start Lesson 03 until Lesson 02 has been reviewed.
